In [ ]:
# Imports 

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# Définition des chemins

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "outputs" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw data:", RAW_DIR)

Project root: c:\Users\dmmaz\GitHub\brussels-north-social-health-needs-lab
Raw data: c:\Users\dmmaz\GitHub\brussels-north-social-health-needs-lab\data\raw


In [ ]:
# Chargement IBSA

file_path = RAW_DIR / "ibsa_population_projections_2026_2035.csv"

population = pd.read_csv(
    file_path,
    sep=";",
    encoding="utf-8"
)

print(population.shape)
population.head()

(3648, 16)


,CD_REFNIS,MUN_FR,MUN_NL,AGE,SEX,POP_2025,PROJ_2026,PROJ_2027,PROJ_2028,PROJ_2029,PROJ_2030,PROJ_2031,PROJ_2032,PROJ_2033,PROJ_2034,PROJ-2035
0,21001,Anderlecht,Anderlecht,Y0,M,796,815,835,857,879,900,902,903,905,907,909
1,21001,Anderlecht,Anderlecht,Y1,M,859,860,837,830,850,830,852,878,901,905,940
2,21001,Anderlecht,Anderlecht,Y2,M,867,814,803,835,837,880,890,877,872,886,874
3,21001,Anderlecht,Anderlecht,Y3,M,880,826,803,776,812,804,820,824,863,862,901
4,21001,Anderlecht,Anderlecht,Y4,M,913,908,852,821,799,844,847,843,818,846,831


In [ ]:
# Vérification des colonnes

population.columns.tolist()

['CD_REFNIS',
 'MUN_FR',
 'MUN_NL',
 'AGE',
 'SEX',
 'POP_2025',
 'PROJ_2026',
 'PROJ_2027',
 'PROJ_2028',
 'PROJ_2029',
 'PROJ_2030',
 'PROJ_2031',
 'PROJ_2032',
 'PROJ_2033',
 'PROJ_2034',
 'PROJ-2035']

In [5]:
population["MUN_FR"].unique()

<ArrowStringArray>
[           'Anderlecht',             'Auderghem', 'Berchem-Sainte-Agathe',
             'Bruxelles',             'Etterbeek',                 'Evere',
                'Forest',             'Ganshoren',               'Ixelles',
                 'Jette',            'Koekelberg',  'Molenbeek-Saint-Jean',
          'Saint-Gilles', 'Saint-Josse-ten-Noode',            'Schaerbeek',
                 'Uccle',   'Watermael-Boitsfort',  'Woluwe-Saint-Lambert',
   'Woluwe-Saint-Pierre']
Length: 19, dtype: str

In [ ]:
# Nettoyage minimal

population = population.rename(
    columns={"PROJ-2035": "PROJ_2035"}
)

population["AGE_NUM"] = (
    population["AGE"]
    .str.extract(r"(\d+)")[0]
    .astype(int)
)

population.head()

,CD_REFNIS,MUN_FR,MUN_NL,AGE,SEX,POP_2025,PROJ_2026,PROJ_2027,PROJ_2028,PROJ_2029,PROJ_2030,PROJ_2031,PROJ_2032,PROJ_2033,PROJ_2034,PROJ_2035,AGE_NUM
0,21001,Anderlecht,Anderlecht,Y0,M,796,815,835,857,879,900,902,903,905,907,909,0
1,21001,Anderlecht,Anderlecht,Y1,M,859,860,837,830,850,830,852,878,901,905,940,1
2,21001,Anderlecht,Anderlecht,Y2,M,867,814,803,835,837,880,890,877,872,886,874,2
3,21001,Anderlecht,Anderlecht,Y3,M,880,826,803,776,812,804,820,824,863,862,901,3
4,21001,Anderlecht,Anderlecht,Y4,M,913,908,852,821,799,844,847,843,818,846,831,4


In [ ]:
# Contrôle qualité

year_columns = (
    ["POP_2025"]
    + [f"PROJ_{year}" for year in range(2026, 2036)]
)

assert population["CD_REFNIS"].nunique() == 19
assert population["SEX"].isin(["M", "F"]).all()
assert population[year_columns].notna().all().all()
assert (population[year_columns] >= 0).all().all()

print("Basic quality checks passed.")

Basic quality checks passed.


In [ ]:
# Sélection du territoire

core_municipalities = [
    "Berchem-Sainte-Agathe",
    "Ganshoren",
    "Jette",
    "Koekelberg",
]

context_municipalities = core_municipalities + ["Bruxelles"]

north_context = population[
    population["MUN_FR"].isin(context_municipalities)
].copy()

north_context["MUN_FR"].value_counts()

MUN_FR
Berchem-Sainte-Agathe    192
Bruxelles                192
Ganshoren                192
Jette                    192
Koekelberg               192
Name: count, dtype: int64

## Limite géographique

Le Bassin Nord de Bruxelles comprend Berchem-Sainte-Agathe,
Ganshoren, Jette, Koekelberg, Laeken, Neder-Over-Heembeek et Haren.

Les projections démographiques de l’IBSA sont disponibles au niveau communal.
La commune de Bruxelles ne peut donc pas être assimilée à la seule partie
de Bruxelles-Ville appartenant au Bassin Nord.

Par conséquent :

- les quatre communes entièrement incluses dans le Bassin Nord sont analysées comme zone principale ;
- Bruxelles-Ville est présentée uniquement comme information de contexte ;
- aucune estimation globale de la population future du Bassin Nord n’est calculée à partir de ce seul jeu de données.

In [ ]:
# Population totale des territoires

municipality_totals = (
    north_context
    .groupby("MUN_FR")[year_columns]
    .sum()
    .reset_index()
)

municipality_totals

,MUN_FR,POP_2025,PROJ_2026,PROJ_2027,PROJ_2028,PROJ_2029,PROJ_2030,PROJ_2031,PROJ_2032,PROJ_2033,PROJ_2034,PROJ_2035
0,Berchem-Sainte-Agathe,25803,25863,25834,25853,25874,25900,25856,25818,25796,25776,25771
1,Bruxelles,198314,198578,199097,199266,199341,199351,199396,199868,200114,200371,200613
2,Ganshoren,25693,25768,25777,25811,25830,25853,25835,25816,25799,25796,25790
3,Jette,54390,54391,54291,54213,54122,54041,53910,53806,53709,53609,53506
4,Koekelberg,22979,23046,23210,23325,23406,23490,23496,23614,23704,23801,23894
